# 实验指南：最小预训练闭环 (Pre-train Minimal)

本实验按顺序带你完成一次可复现的最小训练闭环，重点解决常见的"缓存分词器不一致"报错。

## 你将依次完成的 5 步

1. 环境准备：确保 Python 执行路径在项目根目录。
2. 清理缓存并生成分词器：删除旧缓存，再调用 `download()` 与 `encode_corpus()` 重建数据与 tokenizer。
3. 理解并覆盖配置：读取 `pretrain_tiny.yaml`，理解关键参数并准备覆盖项。
4. 启动训练程序：传入正确的 `data.tokenizer.kind` 与 `train.max_steps` 完成快速训练。
5. 推理验证：使用生成的 `ckpt.pt` 进行文本生成验证。

---

## Step 1. 环境准备（切换执行路径到项目根目录）

本单元会自动检测当前路径；如果你在 `docs/` 下打开 Notebook，会自动切回仓库根目录。

In [1]:
import os
import sys
from pathlib import Path

# 自动向上查找项目根目录（以 pyproject.toml 作为标志文件）
cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").exists()), None)

if repo_root is None:
    # 自动定位失败时，给出手动切换与检测方式
    print("⚠️ 未自动定位到项目根目录。")
    print("请手动执行: %cd d:/codingProgram/LLM-Walk-Through")
    print("然后检测: Path('pyproject.toml').exists() 和 Path('core').exists()")
    raise RuntimeError("请先切换到仓库根目录后再继续。")

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("当前工作目录:", Path.cwd())
print("根目录检测:", (Path("pyproject.toml").exists(), Path("core").exists()))

当前工作目录: D:\codingProgram\LLM-Walk-Through
根目录检测: (True, True)


## Step 2. 清理缓存并生成训练数据子集

使用 Hugging Face 上的标准中文数据集 `opencsg/smoltalk-chinese`，只抽取一个很小的子集来跑通流程。

通过 `max_samples` 与 `max_chars` 限制规模，保证本地机器也能快速完成实验。

In [ ]:
from pathlib import Path
from data.download import download

repo_id = "opencsg/smoltalk-chinese"
tokenizer_kind = "bpe"  # 可选: bpe / byte_bpe / wordpiece / unigram
vocab_size = 2048
max_samples = 2000
max_chars = 2_000_000

cache_dir = Path("data/cache/smoltalk_chinese_small")

# Step 2a: 下载语料（幂等——已下载的 shard 不会重复拉取）
snapshot_dir = download(
    repo_id=repo_id,
    local_dir=cache_dir,
    hf_endpoint="https://hf-mirror.com",
)
total_size_kb = sum(p.stat().st_size for p in snapshot_dir.rglob("*.parquet")) / 1024
print(f"\n数据目录: {snapshot_dir}  ({total_size_kb:.1f} KB)")

In [ ]:
from core.tokenizer import build_tokenizer, load_tokenizer
from data.encode import iter_texts, encode_corpus

# Step 2b: 训练分词器 & 编码为二进制
tokenizer_path = cache_dir / "tokenizer.json"

if tokenizer_path.exists():
    tok = load_tokenizer(tokenizer_path)
    print(f"复用已有分词器 kind={tok.KIND!r}")
else:
    sample_texts = []
    for text in iter_texts(cache_dir, max_chars=max_chars):
        sample_texts.append(text)
    tok_cls = type(build_tokenizer(tokenizer_kind))
    tok = tok_cls.train("\n\n".join(sample_texts), vocab_size=vocab_size, verbose=True)
    tok.save(tokenizer_path)
    print(f"训练完成: kind={tok.KIND} vocab={tok.vocab_size}")

result = encode_corpus(cache_dir=cache_dir, tokenizer=tok, val_ratio=0.1)

print("\n分词 & 编码完成:")
for k, v in result.items():
    print(f"- {k}: {v}")

## Step 3. 理解并覆盖配置文件

主配置文件：[`configs/train/pretrain_tiny.yaml`](../../configs/train/pretrain_tiny.yaml)。

这份配置现在默认指向：
- `data.hf.repo_id = opencsg/smoltalk-chinese`
- `data.hf.max_samples` / `data.hf.max_chars`：控制本地实验子集规模
- `data.tokenizer.kind`：决定要训练哪种分词器
- `train.max_steps`：决定本次演示训练多久

建议先读取配置，再只覆盖最关键的几项参数。

In [ ]:
from omegaconf import OmegaConf

cfg = OmegaConf.load("configs/train/pretrain_tiny.yaml")

print("配置文件已加载: configs/train/pretrain_tiny.yaml")
print("\n关键字段预览:")
print("- data.hf.repo_id:", cfg.data.hf.repo_id)
print("- data.hf.max_samples:", cfg.data.hf.max_samples)
print("- data.tokenizer.kind:", cfg.data.tokenizer.kind)
print("- train.max_steps:", cfg.train.max_steps)
print("- train.batch_size:", cfg.train.batch_size)
print("- train.block_size:", cfg.train.block_size)

max_steps = 2000
print("\n本次覆盖参数:")
print("- data.tokenizer.kind =", tokenizer_kind)
print("- train.max_steps =", max_steps)

## Step 4. 启动训练程序

直接在 kernel 内调用 `train.pretrain.train(cfg)`，可获得 **tqdm 实时进度条**（需 `ipywidgets`）。

若 VS Code Jupyter 中进度条仍无法实时显示，可复制 cell 输出中的命令，在终端直接运行。


In [10]:
from core.utils.config import load_config
from train.pretrain import train as run_train

# 构建覆盖项并加载配置
overrides = [
    f"train.max_steps={max_steps}",
    f"data.tokenizer.kind={tokenizer_kind}",
]
train_cfg = load_config("configs/train/pretrain_tiny.yaml", overrides=overrides)

print(f"开始训练: repo_id={repo_id}, tokenizer_kind={tokenizer_kind}, max_steps={max_steps}")
run_train(train_cfg)


开始训练: repo_id=opencsg/smoltalk-chinese, tokenizer_kind=bpe, max_steps=2000
[setup] device=cpu dtype=torch.float32 amp=False world_size=1
[data] 语料已存在，跳过下载: data\cache\smoltalk_chinese_small\input.txt
[data] 复用已有分词器 kind='bpe': data\cache\smoltalk_chinese_small\tokenizer.json
[data] bin 文件已存在，跳过重新编码。
[setup] model params = 0.32M


训练:   0%|          | 0/2001 [00:00<?, ?step/s]

[eval] step=0 train=8.1025 val=8.1144
[eval] step=50 train=5.0677 val=4.9485
[eval] step=100 train=4.4631 val=4.5085
[eval] step=150 train=4.2806 val=4.2883
[eval] step=200 train=4.0728 val=4.1230
[eval] step=250 train=3.8919 val=3.9559
[eval] step=300 train=3.7725 val=3.8511
[eval] step=350 train=3.7472 val=3.7373
[eval] step=400 train=3.6074 val=3.7254
[eval] step=450 train=3.6114 val=3.5726
[eval] step=500 train=3.3614 val=3.5710
[eval] step=550 train=3.4147 val=3.5157
[eval] step=600 train=3.3268 val=3.4329
[eval] step=650 train=3.2026 val=3.3927
[eval] step=700 train=3.1862 val=3.3299
[eval] step=750 train=3.0912 val=3.3045
[eval] step=800 train=3.1627 val=3.2531
[eval] step=850 train=3.1813 val=3.2362
[eval] step=900 train=3.1277 val=3.1747
[eval] step=950 train=3.0546 val=3.1376
[eval] step=1000 train=2.9853 val=3.2055
[eval] step=1050 train=3.0776 val=3.2104
[eval] step=1100 train=3.0116 val=3.1515
[eval] step=1150 train=2.9850 val=3.0898
[eval] step=1200 train=2.9066 val=3.129

## Step 5. 推理验证

训练完成后，检查是否生成：
- 模型权重：`runs/smoltalk_chinese_small/ckpt.pt`
- 分词器文件：`data/cache/smoltalk_chinese_small/tokenizer.json`

然后使用 `scripts.generate` 做一次最小推理验证。由于当前训练语料是中文数据，prompt 也建议换成中文。

In [11]:
!python -m scripts.generate \
    --checkpoint runs/smoltalk_chinese_small/ckpt.pt \
    --tokenizer data/cache/smoltalk_chinese_small/tokenizer.json \
    --prompt "你好，请介绍一下你自己：" \
    --max-new-tokens 100

你好，请介绍一下你自己：
每句号回答：2、10字

user: 请给出一个句话描述，每个句以下列出三个关的一个字，每段落必须不是一句子信息。
assistant: 1. 1. 数


---

## 常见问题排查

- 若训练前报 tokenizer 不匹配：回到 Step 2 重新执行清理与 `download()` + `encode_corpus()`。
- 若 HF 下载较慢：检查网络，或调整 `hf_endpoint`。
- 若无法自动识别文本字段：在配置里显式设置 `data.hf.text_field`。
- 若路径错误：回到 Step 1，确认当前工作目录是仓库根目录。

完成以上 5 步后，你就已经跑通了一个完整的“HF 数据子集准备 -> 训练 -> 推理”闭环。